# Capítulo 3 · Esquemas y contratos

Una auditoría de la agenda de la Clínica Veterinaria Naranco encuentra citas que no deberían existir: un sábado, a las tres de la tarde, en el pasado. Se reproduce el incidente con doce peticiones de reserva, siete con trampa, y se prueban cuatro versiones de `reservar_cita`: sin validar, con un esquema más preciso, con los tipos de Pydantic y con un contrato completo que aplica las reglas de la clínica y explica sus errores.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y la clínica

La clínica, sus herramientas, el modelo y el agente son los del capítulo 1, juego 4.

In [ ]:
!pip install -q transformers==5.19.0 pydantic==2.14.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 3: esquemas y contratos. Una auditoría de la agenda encuentra citas que no
deberían existir; se investiga por qué y se corrige con esquemas y validación."""
import copy
import json
import re
import urllib.request
from datetime import date
from pathlib import Path

import torch
from pydantic import BaseModel, Field, ValidationError, field_validator
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def reservar_cita(mascota, veterinario, fecha, hora, motivo):
    """Reserva un hueco libre de la agenda; devuelve la cita o el problema."""
    if mascota not in MASCOTAS:
        return {"error": f"No existe la mascota {mascota}."}
    if veterinario not in VETERINARIOS:
        return {"error": f"No existe el veterinario {veterinario}."}
    if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == veterinario
           and c["estado"] != "cancelada" for c in CITAS):
        return {"error": f"{VETERINARIOS[veterinario]} ya tiene una cita el {fecha} "
                         f"a las {hora}."}
    cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
            "veterinario": veterinario, "mascota": mascota, "motivo": motivo,
            "estado": "pendiente"}
    CITAS.append(cita)
    return cita


def cancelar_cita(cita):
    """Cancela una cita pendiente."""
    for c in CITAS:
        if c["id"] == cita:
            if c["estado"] != "pendiente":
                return {"error": f"La cita {cita} está «{c['estado']}»."}
            c["estado"] = "cancelada"
            return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
    return {"error": f"No existe la cita {cita}."}


def avisar(dueno, texto):
    """Envía un aviso al dueño (aquí se guarda en una lista)."""
    if dueno not in DUENOS:
        return {"error": f"No existe el cliente {dueno}."}
    AVISOS.append({"dueno": dueno, "texto": texto})
    return {"enviado": True, "a": DUENOS[dueno]["nombre"],
            "por": DUENOS[dueno]["avisos"]}


def edad(nacimiento):
    """Años cumplidos el día de los datos."""
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    """Lo esencial de una mascota, sin el historial completo."""
    m = MASCOTAS[mid]
    return {"id": mid, "nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida",
            "dueno": {"id": m["dueno"], "nombre": DUENOS[m["dueno"]]["nombre"]}}


def vacunas_con_estado(mid):
    """Cada vacuna con su próxima fecha y si ya está vencida."""
    return [{"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"}
            for v in VACUNAS[mid]]


def ficha_cliente_3(id_cliente):
    if id_cliente not in DUENOS:
        return {"error": f"No existe el cliente {id_cliente}. Búscalo por nombre con "
                         "buscar(tabla='duenos', campo='nombre', valor=...)."}
    pendientes = [f for f in FACTURAS if f["dueno"] == id_cliente and not f["pagada"]]
    return {"id": id_cliente, "nombre": DUENOS[id_cliente]["nombre"],
            "avisos_por": DUENOS[id_cliente]["avisos"],
            "mascotas": [{"id": m, "nombre": d["nombre"], "especie": d["especie"]}
                         for m, d in MASCOTAS.items() if d["dueno"] == id_cliente],
            "facturas_sin_pagar": [{"id": f["id"], "fecha": f["fecha"], "total": f["total"]}
                                   for f in pendientes],
            "total_sin_pagar": round(sum(f["total"] for f in pendientes), 2)}


def agenda_3(fecha):
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(fecha)):
        return {"error": f"La fecha «{fecha}» no es válida; usa AAAA-MM-DD."}
    citas = [c for c in CITAS if c["fecha"] == fecha and c["estado"] != "cancelada"]
    return {"fecha": fecha, "citas": [f"{c['id']} {c['hora']} {c['veterinario']} "
                                      f"{MASCOTAS[c['mascota']]['nombre']} ({c['mascota']})"
                                      for c in citas],
            "veterinarios": VETERINARIOS,
            "horas_libres": {v: [h for h in HORAS if not any(
                c["hora"] == h and c["veterinario"] == v for c in citas)]
                for v in VETERINARIOS}}


def localizar(mascota, dueno=""):
    """El id de la mascota a partir de su nombre (y el del dueño, si hay varias)."""
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(mascota) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        return None, {"error": f"No hay ninguna mascota llamada «{mascota}»"
                               + (f" de {dueno}" if dueno else "") + " en la clínica."}
    if len(ids) > 1:
        return None, {"error": f"Hay {len(ids)} mascotas llamadas «{mascota}»; indica el "
                               "nombre del dueño.",
                      "mascotas": [resumen_mascota(m) for m in ids]}
    return ids[0], None


def veterinario_de(nombre):
    """El id del veterinario a partir de su nombre o de su id."""
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or \
                sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def buscar_mascota(nombre, dueno=""):
    mid, error = localizar(nombre, dueno)
    return error or resumen_mascota(mid)


def mascotas_de(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{"nombre": DUENOS[d]["nombre"], "mascotas": [
        {"nombre": x["nombre"], "especie": x["especie"], "raza": x["raza"]}
        for m, x in MASCOTAS.items() if x["dueno"] == d]} for d in ids]}


def vacunas(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(),
                     "vacunas": vacunas_con_estado(mid)}


def historial(mascota, ultimas=3, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
        {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
        for v in HISTORIAL[mid][-int(ultimas):]]}


def dosis(medicamento, mascota, dueno=""):
    med = FARMACIA.get(sin_tildes(medicamento))
    if med is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: "
                         + ", ".join(FARMACIA) + "."}
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    m = MASCOTAS[mid]
    avisos = []
    if sin_tildes(medicamento) in [sin_tildes(a) for a in m["alergias"]]:
        avisos.append(f"{m['nombre']} es alérgico a {medicamento}: no recetar.")
    if m["especie"] not in med["especies"]:
        avisos.append(f"{medicamento} no está indicado en {m['especie']}s.")
    return {"medicamento": sin_tildes(medicamento), "mascota": m["nombre"],
            "peso_kg": m["peso_kg"], "dosis_mg_por_kg": med["dosis_mg_kg"],
            "dosis_mg_por_toma": round(med["dosis_mg_kg"] * m["peso_kg"], 2),
            "existencias": med["existencias"],
            "reposicion": med.get("reposicion", "sin fecha") if not med["existencias"] else None,
            "avisos": avisos or "ninguno"}


def huecos_libres(fecha, veterinario=""):
    r = agenda_3(fecha)
    if "error" in r:
        return r
    vets = [veterinario_de(veterinario)] if veterinario else list(VETERINARIOS)
    if vets == [None]:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: r["horas_libres"][v]
                                             for v in vets}}


def citas_de(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
        {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
         "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
        for c in CITAS if c["mascota"] == mid]}


def cuentas(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{k: v for k, v in ficha_cliente_3(d).items() if k != "id"}
                         for d in ids]}


def reservar(mascota, veterinario, fecha, hora, motivo, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    vet = veterinario_de(veterinario)
    if vet is None:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    r = reservar_cita(mid, vet, fecha, hora, motivo)
    if "error" in r:
        return r
    return {"reservada": r["id"], "mascota": MASCOTAS[mid]["nombre"],
            "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}


def avisar_dueno_de(mascota, texto, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or avisar(MASCOTAS[mid]["dueno"], texto)


JUEGO_TAREAS = {"buscar_mascota": buscar_mascota, "mascotas_de": mascotas_de,
                "vacunas": vacunas, "historial": historial, "dosis": dosis,
                "huecos_libres": huecos_libres, "citas_de": citas_de, "cuentas": cuentas,
                "reservar_cita": reservar, "cancelar_cita": cancelar_cita,
                "avisar_dueno": avisar_dueno_de}

In [ ]:
def herramienta(nombre, descripcion, parametros):
    """Esquema de una herramienta; los parámetros sin valor por defecto son obligatorios."""
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object",
                       "required": [p for p, (_, oblig) in parametros.items() if oblig],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parametros.items()}}}}


MASCOTA = ("Nombre de la mascota.", True)


DUENO = ("Nombre del dueño; solo si hay varias mascotas con ese nombre.", False)


ESQUEMAS_TAREAS = [
    herramienta("buscar_mascota", "Datos de una mascota buscada por su nombre: especie, "
                "edad, peso actual, alergias y dueño.", {"nombre": MASCOTA, "dueno": DUENO}),
    herramienta("mascotas_de", "Las mascotas de un cliente, buscado por su nombre.",
                {"dueno": ("Nombre del dueño.", True)}),
    herramienta("vacunas", "Vacunas de una mascota: la próxima fecha de cada una y si "
                "está vencida.", {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("historial", "Las últimas visitas de una mascota: fecha, motivo, "
                "tratamiento y peso.", {"mascota": MASCOTA,
                                        "ultimas": ("Cuántas visitas (por defecto, 3).", False),
                                        "dueno": DUENO}),
    herramienta("dosis", "Dosis de un medicamento para una mascota según su peso actual, "
                "con avisos de alergia o especie y las existencias.",
                {"medicamento": ("Nombre del medicamento.", True), "mascota": MASCOTA,
                 "dueno": DUENO}),
    herramienta("huecos_libres", "Horas libres de la agenda en un día, de un veterinario o "
                "de todos.", {"fecha": ("AAAA-MM-DD.", True),
                              "veterinario": ("Nombre del veterinario; vacío para todos.", False)}),
    herramienta("citas_de", "Citas de una mascota (pasadas y pendientes), con su id.",
                {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("cuentas", "Facturas sin pagar de un cliente, buscado por su nombre, y el "
                "total que debe.", {"dueno": ("Nombre del dueño.", True)}),
    herramienta("reservar_cita", "Reserva una cita en una hora libre.",
                {"mascota": MASCOTA, "veterinario": ("Nombre del veterinario.", True),
                 "fecha": ("AAAA-MM-DD.", True), "hora": ("HH:MM, en punto o y media.", True),
                 "motivo": ("Motivo de la cita.", True), "dueno": DUENO}),
    herramienta("cancelar_cita", "Cancela una cita pendiente.",
                {"cita": ("Id de la cita, como CT-0026 (lo da citas_de).", True)}),
    herramienta("avisar_dueno", "Envía un aviso al dueño de una mascota, por correo o SMS "
                "según prefiera.", {"mascota": MASCOTA, "texto": ("Texto del aviso, en "
                                                                   "español.", True),
                                     "dueno": DUENO}),
]


SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def ejecutar(llamada, funciones):
    """Ejecuta una llamada si la herramienta existe y los argumentos encajan."""
    funcion = funciones.get(llamada["name"])
    if funcion is None:
        return {"error": f"No existe la herramienta {llamada['name']}."}
    try:
        return funcion(**llamada["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Argumentos no válidos: {e}"}


def agente(tarea, funciones, esquemas, max_pasos=8, ver=False):
    """Pensar, actuar, observar y repetir. Cuenta los tokens de los resultados."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, esquemas)
        llamadas = leer_llamadas(respuesta)[:1]            # una cada vez
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "tokens_salida": salida,
                      "pensamiento": pensamiento, "llamadas": []})
        if not llamadas:
            traza[-1]["respuesta"] = respuesta
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        if ver and pensamiento:
            print(f"[{paso}] {pensamiento}")
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = ejecutar(llamada, funciones)
            texto = json.dumps(resultado, ensure_ascii=False)
            traza[-1]["llamadas"].append({**llamada, "resultado": resultado,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def citas_nuevas():
    return [c for c in CITAS if c["id"] not in {x["id"] for x in CITAS0}]

## 2. La auditoría

Las reglas de una cita, escritas en Python. Con la herramienta del capítulo 1, una reserva en sábado entra en la agenda sin ningún error.

In [ ]:
def es_valida(cita):
    """¿Respeta la cita las reglas de la clínica? Sirve para auditar la agenda."""
    try:
        dia = date.fromisoformat(cita["fecha"])
    except (TypeError, ValueError):
        return False
    return (dia > HOY and dia.weekday() < 5 and cita["hora"] in HORAS
            and cita["veterinario"] in VETERINARIOS and cita["mascota"] in MASCOTAS
            and not any(c is not cita and c["fecha"] == cita["fecha"] and c["hora"] == cita["hora"]
                        and c["veterinario"] == cita["veterinario"] and c["estado"] != "cancelada"
                        for c in CITAS))


def auditar():
    """Las citas nuevas de la agenda que no respetan las reglas."""
    return [c for c in citas_nuevas() if not es_valida(c)]


def cargar_datos():
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS
    global VETERINARIOS, HORAS
    MASCOTAS, DUENOS = descargar("mascotas.json"), descargar("duenos.json")
    HISTORIAL, VACUNAS = descargar("historial.json"), descargar("vacunas.json")
    CITAS0, FARMACIA = descargar("citas.json"), descargar("farmacia.json")
    FACTURAS, VETERINARIOS = descargar("facturas.json"), descargar("veterinarios.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()

In [ ]:
cargar_datos()
print(reservar("Max", "Fano", "2026-10-24", "10:00", "vacunación"))
print(auditar())

## 3. Un esquema más preciso

In [ ]:
DIAS = "Consulta de lunes a viernes, de 09:00 a 13:30 y de 16:00 a 18:30, cada media hora."


ESQUEMA_PRECISO = {"type": "function", "function": {
    "name": "reservar_cita",
    "description": "Reserva una cita en una hora libre. " + DIAS,
    "parameters": {"type": "object",
                   "required": ["mascota", "veterinario", "fecha", "hora", "motivo"],
                   "properties": {
                       "mascota": {"type": "string", "description": "Nombre de la mascota."},
                       "veterinario": {"type": "string", "enum": []},
                       "fecha": {"type": "string", "format": "date",
                                 "description": "Día de la cita, AAAA-MM-DD; un día laborable "
                                                "posterior a hoy."},
                       "hora": {"type": "string", "pattern": "^(09|1[0-3]|1[6-8]):(00|30)$",
                                "description": "HH:MM, en punto o y media."},
                       "motivo": {"type": "string", "description": "Motivo de la cita."},
                       "dueno": {"type": "string", "description": "Nombre del dueño; solo si "
                                 "hay varias mascotas con ese nombre."}}}}}

## 4. El contrato con Pydantic

Primero solo los tipos y el formato; después, las reglas de la clínica y errores que explican cómo corregir.

In [ ]:
class ReservaTipos(BaseModel):
    mascota: str = Field(description="Nombre de la mascota.")
    veterinario: str = Field(description="Nombre del veterinario.")
    fecha: date = Field(description="Día de la cita, AAAA-MM-DD.")
    hora: str = Field(pattern=r"^(09|1[0-3]|1[6-8]):(00|30)$", description="HH:MM, en punto o y media.")
    motivo: str = Field(min_length=3, description="Motivo de la cita.")
    dueno: str = Field("", description="Nombre del dueño; solo si hay varias mascotas con ese nombre.")


class Reserva(ReservaTipos):
    @field_validator("veterinario")
    @classmethod
    def veterinario_conocido(cls, v):
        if veterinario_de(v) is None:
            raise ValueError(f"no hay ningún veterinario «{v}»; son "
                             + ", ".join(VETERINARIOS.values()))
        return VETERINARIOS[veterinario_de(v)]

    @field_validator("fecha")
    @classmethod
    def dia_de_consulta(cls, v):
        if v <= HOY:
            raise ValueError(f"el {v} no es posterior a hoy ({HOY}); la reserva tiene que "
                             "ser para un día futuro")
        if v.weekday() >= 5:
            raise ValueError(f"el {v} es {'sábado' if v.weekday() == 5 else 'domingo'} y la "
                             "clínica cierra; abre de lunes a viernes")
        return v

    @field_validator("hora", mode="before")
    @classmethod
    def hora_de_consulta(cls, v):
        m = re.fullmatch(r"(\d{1,2})[:.h]?(\d{2})?", str(v).strip())
        hora = m and f"{int(m.group(1)):02d}:{m.group(2) or '00'}"
        if hora not in HORAS:
            raise ValueError(f"«{v}» no es una hora de consulta; las citas son en punto o y "
                             "media, de 09:00 a 13:30 y de 16:00 a 18:30")
        return hora


def explicar(error):
    """Los errores de Pydantic, escritos para el modelo: qué campo y cómo corregirlo."""
    partes = []
    for e in error.errors():
        if e["type"] == "missing":
            texto = "falta este dato"
        elif e["type"].startswith("date"):
            texto = f"«{e['input']}» no es una fecha válida; usa AAAA-MM-DD, como 2026-10-26"
        else:
            texto = e["msg"].removeprefix("Value error, ")
        partes.append(f"{e['loc'][0] if e['loc'] else 'reserva'}: {texto}")
    return "; ".join(partes)


def reservar_validada(contrato, explicar_errores, **argumentos):
    """Valida la reserva con el modelo de Pydantic antes de tocar la agenda. Con
    explicar_errores, el error dice qué corregir y, si la hora está ocupada, da las horas
    libres de ese día; sin él, es el texto de error de Pydantic."""
    try:
        r = contrato(**argumentos)
    except ValidationError as e:
        return {"error": explicar(e) if explicar_errores else str(e)}
    resultado = reservar(r.mascota, r.veterinario, r.fecha.isoformat(), r.hora, r.motivo, r.dueno)
    if explicar_errores and "ya tiene una cita" in resultado.get("error", ""):
        resultado["horas_libres"] = huecos_libres(r.fecha.isoformat(), r.veterinario)["horas_libres"]
    return resultado


def esquema_de(modelo_pydantic, nombre, descripcion):
    """El esquema de la herramienta, sacado del modelo de Pydantic."""
    parametros = modelo_pydantic.model_json_schema()
    parametros.pop("title", None)
    parametros.pop("description", None)
    for p in parametros["properties"].values():
        p.pop("title", None)
    return {"type": "function", "function": {"name": nombre, "description": descripcion,
                                             "parameters": parametros}}

In [ ]:
for contrato, explicada in ((ReservaTipos, False), (Reserva, True)):
    for argumentos in ({"fecha": "2026-10-24", "hora": "10:00"}, {"fecha": "2026-10-26", "hora": "15:00"},
                       {"fecha": "26/10/2026", "hora": "17:30"}, {"fecha": "2026-10-26", "hora": "9"}):
        reiniciar()
        print(contrato.__name__, argumentos, "→",
              reservar_validada(contrato, explicada, mascota="Max", veterinario="Fano",
                                motivo="vacunación", **argumentos))

In [ ]:
print(json.dumps(esquema_de(Reserva, "reservar_cita", "Reserva una cita."), ensure_ascii=False, indent=1))

## 5. Las doce reservas con las cuatro versiones

In [ ]:
def juego(version):
    """Las herramientas y los esquemas de cada versión; solo cambia reservar_cita."""
    funciones, esquemas = dict(JUEGO_TAREAS), list(ESQUEMAS_TAREAS)
    nuevo = None
    if version == "Esquema preciso":
        nuevo = copy.deepcopy(ESQUEMA_PRECISO)
        nuevo["function"]["parameters"]["properties"]["veterinario"]["enum"] = list(VETERINARIOS.values())
    elif version == "Tipos de Pydantic":
        funciones["reservar_cita"] = lambda **a: reservar_validada(ReservaTipos, False, **a)
        nuevo = esquema_de(ReservaTipos, "reservar_cita", "Reserva una cita en una hora libre.")
    elif version == "Contrato completo":
        funciones["reservar_cita"] = lambda **a: reservar_validada(Reserva, True, **a)
        nuevo = esquema_de(Reserva, "reservar_cita", "Reserva una cita en una hora libre. " + DIAS)
    if nuevo:
        esquemas = [nuevo if e["function"]["name"] == "reservar_cita" else e for e in esquemas]
    return funciones, esquemas


VERSIONES = ["Sin validar", "Esquema preciso", "Tipos de Pydantic", "Contrato completo"]


def nuevas(*esperada):
    """¿Es la única cita nueva la esperada (mascota, veterinario, fecha, hora)?"""
    return [(MASCOTAS[c["mascota"]]["nombre"], c["veterinario"], c["fecha"], c["hora"])
            for c in citas_nuevas()] == [esperada]


def ninguna_y_dice(respuesta, *formas):
    """Ninguna cita nueva, y la respuesta explica el motivo."""
    return not citas_nuevas() and dice(respuesta, *formas)


NO = r"no (es posible|se puede|puedo|hay|está|abre|atiende|existe)|cerrad|ocupad|no válid|inválid"


RESERVAS = [
    ("hora_ocupada", "Reserva para Toby una revisión con el Dr. Pablo Menéndez el jueves 22 a las 9.",
     lambda r: ninguna_y_dice(r, "ocupad", "ya tiene", "disponible", NO)),
    ("nueve_y_cuarto", "Reserva para Rocky una consulta con la Dra. Ana Suárez el lunes 26 a las "
     "nueve y cuarto.",
     lambda r: ninguna_y_dice(r, "en punto", "y media", "media hora", NO)),
    ("sabado", "Reserva para Max una vacunación con la Dra. Lucía Fano el sábado 24 a las 10.",
     lambda r: ninguna_y_dice(r, "sábado", "fin de semana", "lunes a viernes")),
    ("tres_de_la_tarde", "Reserva para Kira una revisión con el Dr. Pablo Menéndez el lunes 26 "
     "a las tres de la tarde.",
     lambda r: ninguna_y_dice(r, "16:00", "horario", NO)),
    ("31_septiembre", "Reserva para Lola una revisión con la Dra. Ana Suárez el 31 de septiembre "
     "a las 10.",
     lambda r: ninguna_y_dice(r, "septiembre", NO)),
    ("pasado", "Reserva para Nube una cura con la Dra. Lucía Fano el lunes 19 a las 10.",
     lambda r: ninguna_y_dice(r, "pasad", "anterior", "ya ha", NO)),
    ("barra", "Reserva para Simba una consulta con el Dr. Pablo Menéndez el 26/10 a las 17:30.",
     lambda r: nuevas("Simba", "V2", "2026-10-26", "17:30")),
    ("normal", "Reserva para Thor una desparasitación con la Dra. Ana Suárez el lunes 26 a las 10:30.",
     lambda r: nuevas("Thor", "V1", "2026-10-26", "10:30")),
    ("cuatro_y_media", "Reserva para Coco una revisión con la Dra. Lucía Fano el lunes 26 a las "
     "cuatro y media de la tarde.",
     lambda r: nuevas("Coco", "V3", "2026-10-26", "16:30")),
    ("veterinario_que_no_existe", "Reserva para Luna una consulta con el Dr. García el lunes 26 "
     "a las 11.",
     lambda r: ninguna_y_dice(r, "García", NO) or ninguna_y_dice(r, "Ana Suárez")
     and dice(r, "Lucía Fano")),
    ("primera_de_la_tarde", "Reserva para Bruno una revisión con la Dra. Ana Suárez el viernes 23, "
     "en la primera hora libre de la tarde.",
     lambda r: nuevas("Bruno", "V1", "2026-10-23", "17:00")),
    ("mediodia", "Reserva para Misha una revisión con el Dr. Pablo Menéndez el lunes 26 a mediodía.",
     lambda r: nuevas("Misha", "V2", "2026-10-26", "12:00")),
]


def resolver(funciones, esquemas, tareas=RESERVAS, max_pasos=8):
    """Resuelve cada reserva con la clínica reiniciada; anota el resultado y la auditoría."""
    filas = []
    for clave, tarea, comprobar in tareas:
        reiniciar()
        respuesta, traza = agente(tarea, funciones, esquemas, max_pasos)
        llamadas = [l for p in traza for l in p["llamadas"]]
        filas.append({
            "tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
            "pasos": len(traza), "llamadas": [(l["name"], l["arguments"]) for l in llamadas],
            "rechazadas": sum(l["name"] == "reservar_cita" and "error" in l["resultado"]
                              for l in llamadas),
            "citas_nuevas": citas_nuevas(), "invalidas": auditar(), "traza": traza})
    return filas


def resumen(filas):
    return {"bien": sum(f["bien"] for f in filas),
            "citas_invalidas": sum(len(f["invalidas"]) for f in filas),
            "citas_validas": sum(len(f["citas_nuevas"]) - len(f["invalidas"]) for f in filas),
            "reservas_rechazadas": sum(f["rechazadas"] for f in filas),
            "pasos": sum(f["pasos"] for f in filas)}

In [ ]:
reiniciar()
agente(RESERVAS[2][1], *juego("Sin validar"), ver=True)
print("Auditoría:", auditar())
reiniciar()
agente(RESERVAS[2][1], *juego("Contrato completo"), ver=True)
print("Auditoría:", auditar())

In [ ]:
resultados = {}
for version in VERSIONES:
    resultados[version] = resolver(*juego(version))
    print(f"{version:24s}", resumen(resultados[version]))
    for f in resultados[version]:
        for c in f["invalidas"]:
            print(f"    inválida en {f['tarea']}: {c['fecha']} {c['hora']} {c['veterinario']}")

## Ejercicios

1. Explica con tus palabras la diferencia entre un esquema, un tipo y una regla del negocio. Pon un ejemplo de cada uno en la reserva de una cita.

2. Imprime el esquema que genera `esquema_de(Reserva, …)` y compáralo con el de la versión 2. ¿Qué tiene uno que no tenga el otro? ¿Cuántos tokens ocupa cada uno?

3. Crea a mano objetos `Reserva` con estas horas: «9», «9:00», «09.30», «13:45», «18:30» y «nueve». ¿Cuáles acepta y en qué las convierte?

4. Añade a `Reserva` una regla nueva: las vacunaciones solo se dan por la mañana. ¿Qué tarea escribirías para probarla? ¿Cómo debería ser el mensaje de error?

5. Ejecuta la versión 3 con los errores explicados, sin las reglas del negocio. ¿Cuánto de la mejora se debe a los mensajes y cuánto a las reglas?

6. Haz que `huecos_libres` use el mismo contrato que la reserva, para que no ofrezca horas los sábados ni en el pasado. ¿Cambia algo en las doce reservas?

7. Escribe un contrato para `avisar_dueno`: el texto no puede estar vacío, ni pasar de 300 caracteres, ni contener un número de teléfono. Pruébalo con tres avisos.

8. En la versión sin validar, el agente avisó al dueño de Nube por su cuenta, sin que nadie se lo pidiera, al ver que esa hora no estaba libre. ¿Qué parte del sistema debería impedir esas acciones no pedidas: el mensaje de sistema, la herramienta o el bucle?

9. Prueba la versión del contrato completo con Qwen3-1.7B, sin la línea «Thought:» del mensaje de sistema. ¿Llega alguna cita inválida a la agenda? ¿Por qué no?

10. **Proyecto.** Escribe el contrato completo de `cancelar_cita`: la cita tiene que existir, estar pendiente y no ser de hoy (las de hoy se cancelan por teléfono). Diseña cinco tareas, mide la herramienta con y sin el contrato y escribe un breve informe.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). Documentación de [validadores](https://docs.pydantic.dev/latest/concepts/validators/) y de [esquemas JSON](https://docs.pydantic.dev/latest/concepts/json_schema/) de Pydantic. [JSON Schema](https://json-schema.org/understanding-json-schema/). Lecciones 13.04, 13.05 y 13.28 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.